                                                        ASSIGNMENT NO. :- 9
TITLE :- Develop an AI model to analyze social media sentiment trends for financial markets.
NAME :-Harshada shitole
ROLL NO. :-23107122

In [1]:
pip install pandas numpy scikit-learn nltk


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

/home/admin1/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/admin1/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
data = pd.read_csv("stock_tweets.csv")

print(data.head())

                        Date  \
0  2022-09-29 23:41:16+00:00   
1  2022-09-29 23:24:43+00:00   
2  2022-09-29 23:18:08+00:00   
3  2022-09-29 22:40:07+00:00   
4  2022-09-29 22:27:05+00:00   

                                               Tweet Stock Name Company Name  
0  Mainstream media has done an amazing job at br...       TSLA  Tesla, Inc.  
1  Tesla delivery estimates are at around 364k fr...       TSLA  Tesla, Inc.  
2  3/ Even if I include 63.0M unvested RSUs as of...       TSLA  Tesla, Inc.  
3  @RealDanODowd @WholeMarsBlog @Tesla Hahaha why...       TSLA  Tesla, Inc.  
4  @RealDanODowd @Tesla Stop trying to kill kids,...       TSLA  Tesla, Inc.  


In [5]:
print(data.columns)

Index(['Date', 'Tweet', 'Stock Name', 'Company Name'], dtype='object')


In [6]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    
    return " ".join(words)

data['clean_tweet'] = data['Tweet'].astype(str).apply(clean_text)

print(data[['Tweet','clean_tweet']].head())

                                               Tweet  \
0  Mainstream media has done an amazing job at br...   
1  Tesla delivery estimates are at around 364k fr...   
2  3/ Even if I include 63.0M unvested RSUs as of...   
3  @RealDanODowd @WholeMarsBlog @Tesla Hahaha why...   
4  @RealDanODowd @Tesla Stop trying to kill kids,...   

                                         clean_tweet  
0  mainstream media done amazing job brainwashing...  
1    tesla delivery estimates around k analysts tsla  
2  even include unvested rsus additional equity n...  
3  realdanodowd wholemarsblog tesla hahaha still ...  
4  realdanodowd tesla stop trying kill kids sad d...  


In [7]:
positive_words = ['good','great','profit','gain','up','bull','growth']
negative_words = ['bad','loss','down','drop','bear','fall']

def get_sentiment(text):
    
    score = 0
    
    for word in positive_words:
        if word in text:
            score += 1
            
    for word in negative_words:
        if word in text:
            score -= 1
    
    if score > 0:
        return "positive"
    elif score < 0:
        return "negative"
    else:
        return "neutral"

data['sentiment'] = data['clean_tweet'].apply(get_sentiment)

print(data[['clean_tweet','sentiment']].head())

                                         clean_tweet sentiment
0  mainstream media done amazing job brainwashing...   neutral
1    tesla delivery estimates around k analysts tsla   neutral
2  even include unvested rsus additional equity n...   neutral
3  realdanodowd wholemarsblog tesla hahaha still ...   neutral
4  realdanodowd tesla stop trying kill kids sad d...   neutral


In [8]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(data['clean_tweet'])
y = data['sentiment']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
model = LogisticRegression()

model.fit(X_train, y_train)

/home/admin1/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [11]:
y_pred = model.predict(X_test)

In [12]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.9397858778389752
              precision    recall  f1-score   support

    negative       0.88      0.58      0.70       797
     neutral       0.94      0.98      0.96     12223
    positive       0.96      0.86      0.90      3139

    accuracy                           0.94     16159
   macro avg       0.92      0.81      0.85     16159
weighted avg       0.94      0.94      0.94     16159



In [13]:
sample = ["Tesla stock will grow a lot this year"]

sample_clean = [clean_text(text) for text in sample]

sample_vector = vectorizer.transform(sample_clean)

prediction = model.predict(sample_vector)

print("Sentiment:", prediction[0])

Sentiment: neutral


In [14]:
sentiment_counts = data['sentiment'].value_counts()

print(sentiment_counts)

sentiment
neutral     60922
positive    15969
negative     3902
Name: count, dtype: int64
